# RAY-IMAGE N5.1 — Shape Diagnostic

**Diagnostic-only experiment. No training.**

Restores the existing N5 and N2 checkpoints, runs the controlled 12-prompt comparison with seed 42 and 50 sampling steps, and produces the real N5.1 geometry report.

**Do not retrain, modify model code, or create a new checkpoint.**

In [ ]:
# 1) GPU/runtime check + clone the exact active branch
!nvidia-smi || true
!rm -rf /content/anime-ai-companion
!git clone -b arena/01a07cdc-anime-ai-companion https://github.com/Rishidev-20thcenturey/anime-ai-companion.git /content/anime-ai-companion

In [ ]:
# 2) Verify the diagnostic module exists
%cd /content/anime-ai-companion
!git rev-parse --short HEAD
!ls -l ray_image/probe_n5_shape.py

In [ ]:
# 3) Mount Drive and restore the existing checkpoints (no training)
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import shutil
n5 = Path('/content/drive/MyDrive/RAY_IMAGE/checkpoints/ray_image_v0_3_n5.pt')
n2 = Path('/content/drive/MyDrive/RAY_IMAGE/checkpoints/ray_image_v0_2_whiten.pt')
assert n5.exists(), f'Missing N5 checkpoint: {n5}'
assert n2.exists(), f'Missing N2 checkpoint: {n2}'
shutil.copy2(n5, '/content/ray_image_v0_3_n5.pt')
shutil.copy2(n2, '/content/ray_image_v0_2_whiten.pt')
print('N5 restored:', n5)
print('N2 restored:', n2)

## 4) Run N5.1

This is the only compute-heavy cell. It performs inference/diagnostics only and **does not train**.

In [ ]:
!rm -rf /content/n5_1 /content/n5_1_report.json
!python -m ray_image.probe_n5_shape \
  --n5-checkpoint /content/ray_image_v0_3_n5.pt \
  --outdir /content/n5_1 \
  --n2-checkpoint /content/ray_image_v0_2_whiten.pt

In [ ]:
# 5) Print the complete machine-readable report
import json
from pathlib import Path
report = Path('/content/n5_1/n5_1_report.json')
assert report.exists(), f'Report not found: {report}'
data = json.loads(report.read_text())
print(json.dumps(data, indent=2))


In [ ]:
# 6) Persist report + generated PNGs to Drive
from pathlib import Path
from shutil import copy2
drive_out = Path('/content/drive/MyDrive/RAY_IMAGE/runs/N5.1')
drive_out.mkdir(parents=True, exist_ok=True)
copy2('/content/n5_1/n5_1_report.json', drive_out / 'n5_1_report_real.json')
img_dir = Path('/content/n5_1')
count = 0
for p in img_dir.rglob('*.png'):
    rel = p.relative_to(img_dir)
    dest = drive_out / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    copy2(p, dest)
    count += 1
print('Persisted report:', drive_out / 'n5_1_report_real.json')
print('Persisted PNGs:', count)


In [ ]:
# 7) Display any generated PNGs directly in Colab for optional visual checking
from IPython.display import display
from PIL import Image
from pathlib import Path
for p in sorted(Path('/content/n5_1').rglob('*.png')):
    print(p.relative_to('/content/n5_1'))
    display(Image.open(p))

## Done

Send the **full JSON output from cell 5** back for analysis. No second 8,000-step training run is needed.